# Celeb-DF++ Dataset Overview

Этот ноутбук показывает базовую проверку распакованного датасета:
- сколько всего видео
- какие есть сплиты (train/val/test и т.д.)
- топ папок по количеству видео
- распределение по расширениям

In [ ]:
from pathlib import Path
import pandas as pd

# При необходимости поменяйте путь под вашу среду
DATA_ROOT = Path('/home/n-salikhova/datasets/deepfake_emotion/Celeb-DF++')
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

print('DATA_ROOT:', DATA_ROOT)
print('exists:', DATA_ROOT.exists())

In [ ]:
def infer_split_from_path(path: Path) -> str:
    parts = [p.lower() for p in path.parts]
    candidates = {'train', 'val', 'valid', 'validation', 'test', 'dev'}
    for p in parts:
        if p in candidates:
            if p == 'valid':
                return 'val'
            return p
    return 'unknown'

rows = []
for p in DATA_ROOT.rglob('*'):
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
        rel = p.relative_to(DATA_ROOT)
        rows.append({
            'video_path': str(p),
            'relative_path': str(rel),
            'suffix': p.suffix.lower(),
            'split': infer_split_from_path(rel),
            'top_level_dir': rel.parts[0] if rel.parts else '.',
        })

videos_df = pd.DataFrame(rows)
videos_df.head()

In [ ]:
if videos_df.empty:
    print('Видео не найдены. Проверьте DATA_ROOT и распаковку.')
else:
    print('Total videos:', len(videos_df))
    print('Unique top-level dirs:', videos_df['top_level_dir'].nunique())
    print('--- Split counts ---')
    display(videos_df['split'].value_counts(dropna=False).rename_axis('split').to_frame('n'))
    print('--- Extension counts ---')
    display(videos_df['suffix'].value_counts(dropna=False).rename_axis('suffix').to_frame('n'))
    print('--- Top-level directory counts ---')
    display(videos_df['top_level_dir'].value_counts(dropna=False).head(50).rename_axis('top_level_dir').to_frame('n'))

In [ ]:
# Если хотите сохранить обзор в CSV
OUT_CSV = Path('../metadata/celeb_dfpp_video_inventory.csv').resolve()
if not videos_df.empty:
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    videos_df.to_csv(OUT_CSV, index=False)
    print('Saved:', OUT_CSV)